# Feature Engineering: Sleep Stages Feature

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')  # go up to scripts/ root
import os
from pathlib import Path

import pandas as pd
from datetime import datetime


from data_helper.clean_data_functions import clean_data_baseline_optimized, clean_data_no_data_days_optimized
from data_helper.download_REDCap_data import download_files_for_records
from plot_helper.descriptive_stats_plot import plot_descriptive_stats
from plot_helper.colors import COLORS

#dataframe for all features
df_features_sleep_stages = pd.DataFrame()

## Download Data from REDCap
(Comment out if not needed)


In [ ]:
download_files_for_records(['all'], "venu3_sleep_summary", "baseline_period_arm_1", "data/sleep_summary")

download_files_for_records(['all'], "venu3_sleep_stage", "baseline_period_arm_1", "data/sleep_stage")

## Create Base DataFrames


In [ ]:
#Get Data where Baseline completed
folder_sleep_summary = Path("../../data/sleep_summary")

#Get merged csv file paths
sleep_summary_files = [f.path for f in os.scandir(folder_sleep_summary) if f.is_file() and f.name.endswith(".csv")]

#create dataframes
df_sleep_summary_raw = pd.concat(
    [pd.read_csv(f) for f in sleep_summary_files],
    ignore_index=True
)

df_sleep_summary_raw['durationInHrs'] = df_sleep_summary_raw['durationInMs'] / (1000 * 60 * 60)


#clean dataframes
##match baseline dates
print(f"Sleep summary data before first cropping: {df_sleep_summary_raw.shape}")
df_sleep_summary_raw = clean_data_baseline_optimized([df_sleep_summary_raw])[0]
print(f"Sleep summary data after first cropping: {df_sleep_summary_raw.shape}")
print(f"------"*20)

#*--Make sure sleep does not start too early as calendarDate == wake up date
#*drop rows where calendarDate is smaller than start visit date plus 1 day or calendarDate is larger than end visit date
df_visit_dates = pd.read_csv("../../data/checks/visit_dates_2026-07-08.csv")
df_visit_dates_v1 = df_visit_dates[df_visit_dates['visit_name'] == 'V1'].copy()

# Convert dates
df_sleep_summary_raw["calendarDate"] = pd.to_datetime(
    df_sleep_summary_raw["calendarDate"],
    errors="coerce"
)

df_visit_dates_v1["start_visit_date"] = pd.to_datetime(
    df_visit_dates_v1["start_visit_date"],
    errors="coerce"
)

df_visit_dates_v1["end_visit_date"] = pd.to_datetime(
    df_visit_dates_v1["end_visit_date"],
    errors="coerce"
)
# Keep only needed visit columns
df_visit_dates_v1 = df_visit_dates_v1[
    ["study_id", "start_visit_date", "end_visit_date"]
].drop_duplicates("study_id")

# Merge visit dates onto sleep summary
df_sleep_summary_raw = df_sleep_summary_raw.merge(
    df_visit_dates_v1,
    on="study_id",
    how="left"
)

start_cutoff = df_sleep_summary_raw["start_visit_date"] + pd.Timedelta(days=1)

mask_keep = (
    (df_sleep_summary_raw["calendarDate"] >= start_cutoff) &
    (df_sleep_summary_raw["calendarDate"] <= df_sleep_summary_raw["end_visit_date"])
)
df_sleep_summary_raw = df_sleep_summary_raw[mask_keep].copy()
print(f"Sleep summary data after second cropping: {df_sleep_summary_raw.shape}")
display(df_sleep_summary_raw.head())
print("------"*20)


##drop days with no data
df_sleep_summary_raw = clean_data_no_data_days_optimized([df_sleep_summary_raw])[0]
print(f"Sleep summary data after dropping days with no data: {df_sleep_summary_raw.shape}")
display(df_sleep_summary_raw.head())
print("------"*20)


#get number of patients
n_sleep_summary_raw = df_sleep_summary_raw['study_id'].nunique()
print(f"Number of patients in sleep summary data: {n_sleep_summary_raw}")

display(df_sleep_summary_raw.columns)



## Light Sleep

In [ ]:
df_sleep_stage_f1 = df_sleep_summary_raw[['study_id', 'calendarDate', 'datetime', 'lightSleepDurationInMs']].copy()
df_sleep_stage_f1['lightSleepDurationInHrs'] = df_sleep_stage_f1['lightSleepDurationInMs'] / (1000 * 60 * 60)
df_sleep_stage_f1.head()

df_sleep_stage_f1 = df_sleep_stage_f1.sort_values(by=['study_id']).reset_index(drop=True)
#calculate descriptive stats for each study id
total_light_sleep_duration = df_sleep_stage_f1.groupby('study_id')['lightSleepDurationInHrs'].agg(
    mean_light_sleep_duration='mean',
    median_light_sleep_duration='median',
    std_light_sleep_duration='std',
    skewness_light_sleep_duration=('skew')  # positive = right skew
).reset_index()
display(total_light_sleep_duration.head())

#create plots
fig_f1 = plot_descriptive_stats(
    df = df_sleep_stage_f1,
    col_df = "lightSleepDurationInHrs", 
    df_descriptive_stats=total_light_sleep_duration,
    col_mean="mean_light_sleep_duration",
    col_median="median_light_sleep_duration", 
    n_patients=n_sleep_summary_raw,
    feature="Light Sleep Duration",
    title="Light Sleep Duration [h]",
    tickformat=".2f",
    colors = COLORS
    )
fig_f1.show()


#add sleep duration stats to feature dataframe
df_features_sleep_stages = total_light_sleep_duration[['study_id', 'mean_light_sleep_duration', 'median_light_sleep_duration', 'std_light_sleep_duration']].copy()
display(df_features_sleep_stages.head())

#create daily dataframe
df_features_sleep_stages_daily  = df_sleep_stage_f1[['study_id', 'calendarDate', 'lightSleepDurationInHrs']].copy()
df_features_sleep_stages_daily = df_features_sleep_stages_daily.rename(columns={'lightSleepDurationInHrs': 'daily_light_sleep_duration_hrs'})
display(df_features_sleep_stages_daily.head())



## Deep Sleep

In [ ]:
df_sleep_stage_f2 = df_sleep_summary_raw[['study_id', 'calendarDate', 'datetime', 'deepSleepDurationInMs']].copy()
df_sleep_stage_f2['deepSleepDurationInHrs'] = df_sleep_stage_f2['deepSleepDurationInMs'] / (1000 * 60 * 60)
df_sleep_stage_f2.head()

df_sleep_stage_f2 = df_sleep_stage_f2.sort_values(by=['study_id']).reset_index(drop=True)
#calculate descriptive stats for each study id
total_deep_sleep_duration = df_sleep_stage_f2.groupby('study_id')['deepSleepDurationInHrs'].agg(
    mean_deep_sleep_duration='mean',
    median_deep_sleep_duration='median',
    std_deep_sleep_duration='std',
    skewness_deep_sleep_duration=('skew')  # positive = right skew
).reset_index()
display(total_deep_sleep_duration.head())

#create plots
fig_f2 = plot_descriptive_stats(
    df = df_sleep_stage_f2,
    col_df = "deepSleepDurationInHrs", 
    df_descriptive_stats=total_deep_sleep_duration,
    col_mean="mean_deep_sleep_duration",
    col_median="median_deep_sleep_duration", 
    n_patients=n_sleep_summary_raw,
    feature="Deep Sleep Duration",
    title="Deep Sleep Duration [h]",
    tickformat=".2f",
    colors = COLORS
    )

fig_f2.show()

#add onset stats to feature dataframe
df_features_sleep_stages = df_features_sleep_stages.merge(
    total_deep_sleep_duration[['study_id', 'mean_deep_sleep_duration', 'median_deep_sleep_duration', 'std_deep_sleep_duration']],
    on='study_id',
    how='outer'
)
display(df_features_sleep_stages.head())

#add daily onset to df_features_sleep_stages_daily
df_features_sleep_stages_daily = df_features_sleep_stages_daily.merge(
    df_sleep_stage_f2[['study_id', 'calendarDate', 'deepSleepDurationInHrs',]],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_features_sleep_stages_daily = df_features_sleep_stages_daily.rename(columns={'deepSleepDurationInHrs': 'daily_deep_sleep_duration_hrs'})
display(df_features_sleep_stages_daily.head())



## REM Sleep

In [ ]:
df_sleep_stage_f3 = df_sleep_summary_raw[['study_id', 'calendarDate', 'datetime', 'remSleepInMs']].copy()
df_sleep_stage_f3['remSleepInHrs'] = df_sleep_stage_f3['remSleepInMs'] / (1000 * 60 * 60)
df_sleep_stage_f3.head()

df_sleep_stage_f3 = df_sleep_stage_f3.sort_values(by=['study_id']).reset_index(drop=True)
#calculate descriptive stats for each study id
total_rem_sleep_duration = df_sleep_stage_f3.groupby('study_id')['remSleepInHrs'].agg(
    mean_rem_sleep_duration='mean',
    median_rem_sleep_duration='median',
    std_rem_sleep_duration='std',
    skewness_rem_sleep_duration=('skew')  # positive = right skew
).reset_index()
display(total_rem_sleep_duration.head())

#create plots

fig_f3 = plot_descriptive_stats(
    df = df_sleep_stage_f3,
    col_df = "remSleepInHrs", 
    df_descriptive_stats=total_rem_sleep_duration,
    col_mean="mean_rem_sleep_duration",
    col_median="median_rem_sleep_duration", 
    n_patients=n_sleep_summary_raw,
    feature="REM Sleep Duration",
    title="REM Sleep Duration [h]",
    tickformat=".2f",
    colors = COLORS
    )

fig_f3.show()



#add onset stats to feature dataframe
df_features_sleep_stages = df_features_sleep_stages.merge(
    total_rem_sleep_duration[['study_id', 'mean_rem_sleep_duration', 'median_rem_sleep_duration', 'std_rem_sleep_duration']],
    on='study_id',
    how='outer'
)
display(df_features_sleep_stages.head())

#add daily onset to df_features_sleep_stages_daily
df_features_sleep_stages_daily = df_features_sleep_stages_daily.merge(
    df_sleep_stage_f3[['study_id', 'calendarDate', 'remSleepInHrs',]],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_features_sleep_stages_daily = df_features_sleep_stages_daily.rename(columns={'remSleepInHrs': 'daily_rem_sleep_duration_hrs'})
display(df_features_sleep_stages_daily.head())



## Sleep Score

In [ ]:
df_sleep_stage_f4 = df_sleep_summary_raw[['study_id', 'calendarDate', 'datetime', 'overallSleepScore']].copy()

df_sleep_stage_f4.head()

df_sleep_stage_f4 = df_sleep_stage_f4.sort_values(by=['study_id']).reset_index(drop=True)
#calculate descriptive stats for each study id
total_overall_sleep_score = df_sleep_stage_f4.groupby('study_id')['overallSleepScore'].agg(
    mean_overall_sleep_score='mean',
    median_overall_sleep_score='median',
    std_overall_sleep_score='std',
).reset_index()
display(total_overall_sleep_score.head())

#create plots
fig_f4 = plot_descriptive_stats(
    df = df_sleep_stage_f4,
    col_df = "overallSleepScore", 
    df_descriptive_stats=total_overall_sleep_score,
    col_mean="mean_overall_sleep_score",
    col_median="median_overall_sleep_score", 
    n_patients=n_sleep_summary_raw,
    feature="Overall Sleep Score",
    title="Overall Sleep Score",
    tickformat=".0f",
    bins=dict(start=df_sleep_stage_f4['overallSleepScore'].min(), end=df_sleep_stage_f4['overallSleepScore'].max() + 5, size=5),
    colors = COLORS
    )

fig_f4.show()

#add onset stats to feature dataframe
df_features_sleep_stages = df_features_sleep_stages.merge(
    total_overall_sleep_score[['study_id', 'mean_overall_sleep_score', 'median_overall_sleep_score', 'std_overall_sleep_score']],
    on='study_id',
    how='outer'
)
display(df_features_sleep_stages.head())

#add daily onset to df_features_sleep_stages_daily
df_features_sleep_stages_daily = df_features_sleep_stages_daily.merge(
    df_sleep_stage_f4[['study_id', 'calendarDate', 'overallSleepScore',]],
    on=['study_id', 'calendarDate'],
    how='outer'
)
df_features_sleep_stages_daily = df_features_sleep_stages_daily.rename(columns={'overallSleepScore': 'daily_overall_sleep_score'})
display(df_features_sleep_stages_daily.head())



## Store df_feature_sleep

In [ ]:
#if there already exists a file with same name move it to subfolder "archive"
if not os.path.exists('../../output/1_feature_extraction/archive'):
    os.makedirs('../../output/1_feature_extraction/archive')
files = os.listdir('../../output/1_feature_extraction')
for file in files:
    if file.startswith('df_features_sleep_stages_') and file.endswith('.csv'):
        os.rename(f'../../output/1_feature_extraction/{file}', f'../../output/1_feature_extraction/archive/{file}')
    elif file.startswith('df_features_daily_sleep_stages_') and file.endswith('.csv'):
        os.rename(f'../../output/1_feature_extraction/{file}', f'../../output/1_feature_extraction/archive/{file}')

#store df_feature as csv
date = datetime.now().strftime("%Y-%m-%d")
df_features_sleep_stages.to_csv(f'../../output/1_feature_extraction/df_features_sleep_stages_{date}.csv', index=False)
df_features_sleep_stages_daily.to_csv(f'../../output/1_feature_extraction/df_features_daily_sleep_stages_{date}.csv', index=False)